### her2+dish疊合圖處理:測試演算法

In [1]:
import cv2
import numpy as np
from skimage.color import rgb2hed
from skimage.exposure import rescale_intensity
from stardist.models import StarDist2D
from csbdeep.utils import normalize

In [2]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
if len(tf.config.list_physical_devices('GPU')) > 0:
    print("GPU is available")
    print(tf.config.list_physical_devices('GPU'))
else:
    print("GPU is NOT available. Running on CPU.")

Num GPUs Available:  1
GPU is available
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
def process_her2_dish_image(img_path):
    # 1. 讀取影像
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ---------------------------------------------------------
    # 步驟一：色彩分離 (Color Deconvolution)
    # 將 RGB 轉換為 HED 空間 (Hematoxylin, Eosin/DAB, DAB)
    # scikit-image 的 rgb2hed 針對 H&E 優化，但對 H&DAB (棕色) 效果通常也不錯
    # Channel 0: Hematoxylin (細胞核 - 藍紫色)
    # Channel 2: DAB (HER2 蛋白 - 棕色)
    # ---------------------------------------------------------
    hed = rgb2hed(img_rgb)
    
    # 提取細胞核通道 (Hematoxylin)
    nuclei_channel = hed[:, :, 0]
    # 提取 HER2 膜通道 (DAB)
    her2_membrane_channel = hed[:, :, 2]

    # 正規化這些通道以便後續處理 (轉回 0-255)
    nuclei_img_vis = (rescale_intensity(nuclei_channel, out_range=(0, 255))).astype(np.uint8)
    her2_membrane_vis = (rescale_intensity(her2_membrane_channel, out_range=(0, 255))).astype(np.uint8)

    # ---------------------------------------------------------
    # 步驟二：細胞核分割 (使用分離出來的藍色通道)
    # ---------------------------------------------------------
    # 載入 StarDist 模型
    model = StarDist2D.from_pretrained('2D_versatile_he')
    
    # 因為 StarDist 預設吃 RGB，我們把單色通道疊成 3 層偽裝成 RGB
    # 這裡我們用 trick：把分離出的細胞核圖當作輸入
    # 為了增強效果，可以做一點對比度增強
    img_input = np.stack([nuclei_img_vis]*3, axis=-1)
    labels, _ = model.predict_instances(normalize(img_input))

    # ---------------------------------------------------------
    # 步驟三：定義 HER2 陽性區域 (使用分離出來的棕色通道)
    # ---------------------------------------------------------
    # 設定閾值，抓出棕色夠深的地方 (你需要自己 print 圖出來調這個 100)
    _, her2_positive_mask = cv2.threshold(her2_membrane_vis, 100, 255, cv2.THRESH_BINARY)
    
    # ---------------------------------------------------------
    # 步驟四：訊號偵測 (紅點與黑點) - 回到原圖處理
    # ---------------------------------------------------------
    # 轉 HSV 抓紅黑點 (沿用上一次的方法)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # 黑點 (HER2 Gene)
    mask_black = cv2.inRange(hsv, (0, 0, 0), (180, 255, 60)) 
    
    # 紅點 (CEP17 Gene)
    mask_red1 = cv2.inRange(hsv, (0, 70, 50), (10, 255, 255))
    mask_red2 = cv2.inRange(hsv, (170, 70, 50), (180, 255, 255))
    mask_red = cv2.bitwise_or(mask_red1, mask_red2)

    # ---------------------------------------------------------
    # 步驟五：整合計算
    # ---------------------------------------------------------
    results = []
    
    for nuclei_id in np.unique(labels):
        if nuclei_id == 0: continue # 背景

        # 取得單顆細胞核遮罩
        nucleus_mask = (labels == nuclei_id).astype(np.uint8) * 255
        
        # 擴張細胞核遮罩一點點，因為 HER2 膜是在細胞核外圍
        kernel = np.ones((5,5), np.uint8)
        dilated_mask = cv2.dilate(nucleus_mask, kernel, iterations=1)

        # 檢查這顆細胞是否在 HER2 陽性區域 (檢查擴張後的範圍是否有沾到棕色 mask)
        overlap = cv2.bitwise_and(her2_positive_mask, her2_positive_mask, mask=dilated_mask)
        if cv2.countNonZero(overlap) > 10: # 門檻值：沾到多少棕色算陽性
            
            # 在細胞核範圍內算紅黑點
            # 注意：SISH 訊號通常在細胞核內，所以用原本的 nucleus_mask
            roi_black = cv2.bitwise_and(mask_black, mask_black, mask=nucleus_mask)
            roi_red = cv2.bitwise_and(mask_red, mask_red, mask=nucleus_mask)
            
            # 計算連通區域 (算顆數)
            n_her2 = max(0, cv2.connectedComponents(roi_black)[0] - 1)
            n_cep17 = max(0, cv2.connectedComponents(roi_red)[0] - 1)
            
            results.append({
                "id": nuclei_id,
                "her2_dots": n_her2,
                "cep17_dots": n_cep17,
                "ratio": n_her2 / n_cep17 if n_cep17 > 0 else 0
            })

    return results, nuclei_img_vis, her2_membrane_vis

In [4]:
# 測試用：記得把產出的 nuclei_img_vis 和 her2_membrane_vis 存成圖片看看有沒有分乾淨
results, n_img, h_img = process_her2_dish_image('process/tile_x153600_y59392.tiff')
print(results)
cv2.imwrite('process/output/debug_nuclei.jpg', n_img)
cv2.imwrite('process/output/debug_her2.jpg', h_img)

Found model '2D_versatile_he' for 'StarDist2D'.
5294730/5294730 [==============================] - 0s 0us/step
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
[{'id': 11, 'her2_dots': 0, 'cep17_dots': 1, 'ratio': 0.0}, {'id': 14, 'her2_dots': 0, 'cep17_dots': 1, 'ratio': 0.0}, {'id': 66, 'her2_dots': 0, 'cep17_dots': 1, 'ratio': 0.0}, {'id': 138, 'her2_dots': 0, 'cep17_dots': 1, 'ratio': 0.0}]


True